# Simple Agent with LangGraph and AWS Bedrock

This notebook demonstrates a simple agent using LangGraph with AWS Bedrock and Neo4j.

The code:

- Creates an agent using LangGraph with `ChatBedrockConverse`
- Connects to your Neo4j database
- Defines a `get_graph_schema` tool
- Runs the agent to answer questions about the graph schema

Review and run the code and answer the following questions: 

1. What is the agent's function? 
2. What can it do?
3. How could you extend it?

---

## Setup

Run from CLI first (if not done already):
```bash
./setup-inference-profile.sh haiku
```

Then copy the output ARN and paste it into `INFERENCE_PROFILE_ARN` below.

## 1. Configuration

In [ ]:
#################################################
# CONFIGURATION
#################################################

# AWS Bedrock Configuration
INFERENCE_PROFILE_ARN = "PASTE_YOUR_ARN_HERE"  # <-- PASTE HERE
REGION = "us-west-2"

# Neo4j Configuration
NEO4J_URI = "neo4j+s://your-instance.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "your-password"

#################################################

# Validate configuration
errors = []
if "PASTE" in INFERENCE_PROFILE_ARN or "YOUR" in INFERENCE_PROFILE_ARN:
    errors.append("Paste your inference profile ARN (run ./setup-inference-profile.sh)")
if "your-instance" in NEO4J_URI:
    errors.append("Update NEO4J_URI with your Neo4j Aura connection string")
if "your-password" in NEO4J_PASSWORD:
    errors.append("Update NEO4J_PASSWORD with your Neo4j password")

if errors:
    print("ERROR: Configuration incomplete!")
    for e in errors:
        print(f"  - {e}")
else:
    print("Configuration OK!")

## 2. Install and Verify Packages

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "neo4j",
    "neo4j-graphrag",
    "boto3",
]

print("Required packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install missing packages
%pip install langgraph langchain-aws neo4j-graphrag boto3 -q

## 3. Imports

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.schema import get_schema

from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports successful!")

## 4. Connect to Neo4j

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI, 
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

## 5. Define Tools

In LangGraph, tools are defined as Python functions with the `@tool` decorator. The agent will use the function name and docstring to determine when to execute the tool.

In [ ]:
@tool
def get_graph_schema() -> str:
    """Get the schema of the graph database including node labels, relationships, and properties."""
    return get_schema(driver)

# Define a list of tools for the agent
tools = [get_graph_schema]

print(f"Defined {len(tools)} tool(s): {[t.name for t in tools]}")

> The agent will use the tool's name (`get_graph_schema`) and docstring (`Get the schema of the graph database...`) to determine whether it should execute the tool to resolve a user's query.

---

## 6. Initialize LLM and Create Agent

Create the agent using LangGraph's `create_react_agent` with AWS Bedrock.

In [ ]:
# Initialize LLM using AWS Bedrock
llm = ChatBedrockConverse(
    model=INFERENCE_PROFILE_ARN,
    provider="anthropic",
    region_name=REGION,
    temperature=0,
)

# Create the agent with tools
SYSTEM_PROMPT = """You are a helpful assistant that can answer questions about a graph database schema.
Use the get_graph_schema tool to retrieve the database schema when needed.
Be concise and informative in your responses."""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("Agent created successfully!")

## 7. Run the Agent

Create a query for the agent and run it.

When you run the agent, you will see:

1. The agent's reasoning about which tool to use
2. The context of the database schema (retrieved by the tool)
3. The agent's final response

In [ ]:
def run_agent(question: str):
    """Run the agent with a question and display the response."""
    print(f"User: {question}")
    print("-" * 50)
    
    result = agent.invoke({"messages": [("human", question)]})
    
    # Get the final message
    final_message = result["messages"][-1]
    print(f"\nAssistant: {final_message.content}")
    return result

In [ ]:
# Run the agent
query = "Summarise the schema of the graph database."
result = run_agent(query)

## 8. Experiment

Experiment with the agent, modify the `query` to ask different questions, for example:

* `"How are Products related to other entities?"`
* `"What questions can I answer using this graph database?"`
* `"How does the graph model relate financial documents to risk factors?"`

In [ ]:
# Try a different query
query = "What questions can I answer using this graph database?"
result = run_agent(query)

In [ ]:
# Try another query
query = "How are Companies related to RiskFactors?"
result = run_agent(query)

---

[Move on to the Vector + Graph Agent Notebook](02_vector_graph_agent.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")